# Shoaling Carryover Raw ASD Analysis

Frame-level animal-stimulus distance and 1-minute attraction estimates for continuous (`01k01f`) and bout-like (`02k20f`) stimulus episodes, with carryover checks for continuous episodes following bouts.

In [ ]:
# Load plotting libraries and make project imports work from either the repo root or Analyses/.
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%config InteractiveShellApp.pylab_import_all = False
%matplotlib inline
%reload_ext autoreload
%autoreload 2

cwd = Path.cwd()
REPO_DIR = cwd.parent if cwd.name == 'Analyses' else cwd
os.chdir(REPO_DIR)

import functions.paperFigureProps as pfp
import functions.carryover_effects as ce

# Apply the same visual defaults used by the other selection-shoaling notebooks.
pfp.paper()
sns.set_context('talk')
sns.set_style('ticks')


def plot_sem(values):
    values = pd.Series(values).dropna()
    return values.sem() if len(values) > 1 else np.nan


def summarize_plot_values(df, group_cols, value_col):
    return (
        df.groupby(group_cols, dropna=False, observed=True)[value_col]
        .agg(mean='mean', sem=plot_sem, n='count')
        .reset_index()
    )


def draw_lineplot_with_sem(ax, summary, x_col, y_col, sem_col, split_by=None, marker=None):
    if split_by is None:
        summary = summary.sort_values(x_col)
        ax.plot(summary[x_col], summary[y_col], color='k', marker=marker, linewidth=1.8, label='all')
        ax.fill_between(
            summary[x_col].to_numpy(dtype=float),
            (summary[y_col] - summary[sem_col]).to_numpy(dtype=float),
            (summary[y_col] + summary[sem_col]).to_numpy(dtype=float),
            color='k',
            alpha=0.18,
            linewidth=0,
        )
        return

    for label, data in summary.groupby(split_by, dropna=False, observed=True):
        data = data.sort_values(x_col)
        ax.plot(data[x_col], data[y_col], marker=marker, linewidth=1.8, label=str(label))
        color = ax.lines[-1].get_color()
        ax.fill_between(
            data[x_col].to_numpy(dtype=float),
            (data[y_col] - data[sem_col]).to_numpy(dtype=float),
            (data[y_col] + data[sem_col]).to_numpy(dtype=float),
            color=color,
            alpha=0.18,
            linewidth=0,
        )


def episode_bands(df):
    if df.empty:
        return pd.DataFrame()
    return (
        df.groupby(['epiNr', 'episode', 'episode_type'], observed=True)
        .agg(start_min=('time_min', 'min'), end_min=('time_min', 'max'))
        .reset_index()
        .sort_values('start_min')
    )


def draw_episode_bands(ax, bands, alpha=0.1):
    colors = {'continuous': '#E0A32D', 'bouts': '#3B75AF'}
    for _, band in bands.iterrows():
        ax.axvspan(
            band.start_min,
            band.end_min,
            color=colors.get(band.episode_type, '0.8'),
            alpha=alpha,
            linewidth=0,
        )
        xpos = (band.start_min + band.end_min) / 2
        ax.text(xpos, 0.98, band.episode, transform=ax.get_xaxis_transform(), ha='center', va='top', fontsize=7)


## Settings

In [ ]:
# Metadata and analysis settings; edit these first when changing the cohort or temporal resolution.
metaFolder = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/07_Data/Carlos/Behavior/ShoalingSelection/'
metaFile = 'MetaData_CR.xlsx'
ProcessingDir = Path('//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use/temp/shoaling_selection/ShoalCarryover_temp_processing/')

# These files mirror the slim notebook pattern: processing settings plus cached analysis artifacts.
PROCESSING_SETTINGS_PATH = ProcessingDir / 'processingSettings.csv'
ANALYSIS_SETTINGS_PATH = ProcessingDir / 'carryover_analysis_settings.json'
ASD_FRAME_CACHE_PATH = ProcessingDir / 'asd_frame.csv.gz'
SI_1MIN_CACHE_PATH = ProcessingDir / 'si_1min.csv.gz'

YEAR_FILTER = 2026
EPISODE_MAP = {
    '01k01f': 'continuous',
    '02k20f': 'bouts',
}

# Keep only the first 24 chronological raw episode blocks, matching the existing 2-hour pipeline window.
MAX_EPISODE_BLOCKS = 24
FRAME_STRIDE = 1
WINDOW_SECONDS = 60
SHUFFLE_MIN_SHIFT_SECONDS = 30
N_SHIFT_RUNS = 10
GENOTYPE_COLUMN = 'genotype_norm'

# Escapee fish have genotypes beginning with esc; set True to include them again later.
INCLUDE_ESCAPEE_FISH = False

# Keep False to reuse per-experiment processed caches, like MissingOnly=True in the slim notebook.
FORCE_REPROCESS_RAW_DATA = False


## Load Metadata

In [ ]:
# Read the experiment and animal sheets, then resolve raw position-file paths for each experiment.
info_exp, info_an = ce.read_selection_metadata(Path(metaFolder) / metaFile, year=YEAR_FILTER)
carryover_rows, animal_lookup = ce.prepare_carryover_effect_table(info_exp)

# Store the processing-settings table in the temporary folder, like the slim selection notebook does.
ProcessingDir.mkdir(parents=True, exist_ok=True)
carryover_rows.to_csv(PROCESSING_SETTINGS_PATH, encoding='utf-8')

print(f'Experiment rows with raw position files: {len(carryover_rows)}')
print(f'Processing settings saved to: {PROCESSING_SETTINGS_PATH}')
carryover_rows[['date', 'folder', 'txtPath', 'anNr', 'setup']].head()


## Extract Frame-Level ASD And 1 Min SI

In [ ]:
# This cell loads existing per-experiment processed tables and only reads raw positions for missing caches.
progress_preview = carryover_rows.reset_index()[['index', 'date', 'folder', 'setup', 'txtPath']]
print(f'Files queued for carryover extraction: {len(progress_preview)}')
display(progress_preview)

# Save the analysis settings next to the cached tables so the run is reproducible later.
processed_cache_settings = {
    'year_filter': YEAR_FILTER,
    'episode_map': EPISODE_MAP,
    'max_episode_blocks': MAX_EPISODE_BLOCKS,
    'frame_stride': FRAME_STRIDE,
    'window_seconds': WINDOW_SECONDS,
    'shuffle_min_shift_seconds': SHUFFLE_MIN_SHIFT_SECONDS,
    'n_shift_runs': N_SHIFT_RUNS,
    'include_escapee_fish': INCLUDE_ESCAPEE_FISH,
}
analysis_settings = {
    **processed_cache_settings,
    'genotype_column': GENOTYPE_COLUMN,
    'force_reprocess_raw_data': FORCE_REPROCESS_RAW_DATA,
    'processing_dir': str(ProcessingDir),
    'processing_settings_path': str(PROCESSING_SETTINGS_PATH),
    'asd_frame_cache_path': str(ASD_FRAME_CACHE_PATH),
    'si_1min_cache_path': str(SI_1MIN_CACHE_PATH),
}
with open(ANALYSIS_SETTINGS_PATH, 'w', encoding='utf-8') as f:
    json.dump(analysis_settings, f, indent=2)

asd_frame, si_1min = ce.collect_or_load_carryover_effect_data(
    carryover_rows,
    processing_dir=ProcessingDir,
    animal_info=info_an,
    animal_lookup=animal_lookup,
    episode_map=EPISODE_MAP,
    max_episode_blocks=MAX_EPISODE_BLOCKS,
    window_seconds=WINDOW_SECONDS,
    shuffle_min_shift_seconds=SHUFFLE_MIN_SHIFT_SECONDS,
    n_shift_runs=N_SHIFT_RUNS,
    frame_stride=FRAME_STRIDE,
    include_escapees=INCLUDE_ESCAPEE_FISH,
    force_reprocess=FORCE_REPROCESS_RAW_DATA,
    cache_settings=processed_cache_settings,
)

# Keep combined cohort-level copies for quick manual loading; per-experiment files control skip/reprocess.
asd_frame.to_csv(ASD_FRAME_CACHE_PATH, index=False, compression='gzip')
si_1min.to_csv(SI_1MIN_CACHE_PATH, index=False, compression='gzip')

print(f'Frame ASD rows: {len(asd_frame):,}')
print(f'1 min SI rows: {len(si_1min):,}')
print(f'Analysis settings saved to: {ANALYSIS_SETTINGS_PATH}')
print(f'Combined ASD frame cache saved to: {ASD_FRAME_CACHE_PATH}')
print(f'Combined 1 min SI cache saved to: {SI_1MIN_CACHE_PATH}')
asd_frame.head()


## Analysis 1: Raw ASD Over Time

In [ ]:
# Plot the raw frame-level ASD mean across all animals, with episode labels shown over time.
fig, ax = plt.subplots(figsize=(13, 4))
draw_episode_bands(ax, episode_bands(asd_frame), alpha=0.08)
asd_full_summary = summarize_plot_values(asd_frame, ['frame_global', 'time_min'], 'asd_mm').sort_values('time_min')
draw_lineplot_with_sem(ax, asd_full_summary, 'time_min', 'mean', 'sem')
ax.set_xlabel('Experiment time (min)')
ax.set_ylabel('Animal-stimulus distance (mm)')
ax.set_title('Frame-by-frame ASD across the experiment')
ax.legend(frameon=False)
plt.show()


In [ ]:
# Collapse repeated episodes onto a shared 0-5 minute axis for each stimulus type.
episode_order = list(EPISODE_MAP)
fig, axes = plt.subplots(1, len(episode_order), figsize=(6 * len(episode_order), 4), sharey=True)
axes = np.atleast_1d(axes)

for ax, episode in zip(axes, episode_order):
    data = asd_frame.loc[asd_frame['episode'] == episode]
    summary = summarize_plot_values(data, ['episode_frame', 'episode_time_min'], 'asd_mm').sort_values('episode_time_min')
    draw_lineplot_with_sem(ax, summary, 'episode_time_min', 'mean', 'sem')
    ax.set_title(episode)
    ax.set_xlabel('Episode time (min)')
    ax.set_ylabel('Animal-stimulus distance (mm)')
    ax.legend(frameon=False)

fig.tight_layout()
plt.show()


## Analysis 1c: Raw ASD Split By Genotype

In [ ]:
# Repeat the full-experiment ASD plot with separate traces for each genotype.
fig, ax = plt.subplots(figsize=(13, 4))
draw_episode_bands(ax, episode_bands(asd_frame), alpha=0.08)
asd_genotype_full_summary = summarize_plot_values(
    asd_frame,
    ['frame_global', 'time_min', GENOTYPE_COLUMN],
    'asd_mm',
).sort_values('time_min')
draw_lineplot_with_sem(ax, asd_genotype_full_summary, 'time_min', 'mean', 'sem', split_by=GENOTYPE_COLUMN)
ax.set_xlabel('Experiment time (min)')
ax.set_ylabel('Animal-stimulus distance (mm)')
ax.set_title('Frame-by-frame ASD across the experiment')
ax.legend(frameon=False)
plt.show()


In [ ]:
# Repeat the episode-average ASD plot with genotype split traces.
episode_order = list(EPISODE_MAP)
fig, axes = plt.subplots(1, len(episode_order), figsize=(6 * len(episode_order), 4), sharey=True)
axes = np.atleast_1d(axes)

for ax, episode in zip(axes, episode_order):
    data = asd_frame.loc[asd_frame['episode'] == episode]
    summary = summarize_plot_values(
        data,
        ['episode_frame', 'episode_time_min', GENOTYPE_COLUMN],
        'asd_mm',
    ).sort_values('episode_time_min')
    draw_lineplot_with_sem(ax, summary, 'episode_time_min', 'mean', 'sem', split_by=GENOTYPE_COLUMN)
    ax.set_title(episode)
    ax.set_xlabel('Episode time (min)')
    ax.set_ylabel('Animal-stimulus distance (mm)')
    ax.legend(frameon=False)

fig.tight_layout()
plt.show()


## Analysis 2: 1 Min Shoaling Index

The 1-minute index uses the same form as the existing 5-minute shoaling index: `(shifted ASD - observed ASD) / shifted ASD`. For each original 5-minute episode, the stimulus trace is circularly shifted by at least 30 seconds, shifted ASD is summarized in the original 1-minute windows, and the shifted mean is used as the local null expectation.

In [ ]:
# Show the 1-minute attraction estimate in chronological experiment order.
fig, ax = plt.subplots(figsize=(13, 4))
draw_episode_bands(ax, episode_bands(si_1min), alpha=0.08)
si_over_time_summary = summarize_plot_values(
    si_1min,
    ['minute_start_global', 'time_min'],
    'si_1min',
).sort_values('time_min')
draw_lineplot_with_sem(ax, si_over_time_summary, 'time_min', 'mean', 'sem', marker='o')
ax.axhline(0, color='k', linestyle=':', linewidth=1)
ax.set_xlabel('Experiment time (min)')
ax.set_ylabel('1 min shoaling index')
ax.set_title('1 min shoaling index across the experiment')
ax.legend(frameon=False)
plt.show()


In [ ]:
# Average the 1-minute attraction estimate across repeated episodes of the same type.
fig, ax = plt.subplots(figsize=(7, 4))
si_episode_summary = summarize_plot_values(
    si_1min,
    ['minute_in_episode', 'episode'],
    'si_1min',
).sort_values('minute_in_episode')
draw_lineplot_with_sem(ax, si_episode_summary, 'minute_in_episode', 'mean', 'sem', split_by='episode', marker='o')
ax.axhline(0, color='k', linestyle=':', linewidth=1)
ax.set_xlabel('Minute within 5 min episode')
ax.set_ylabel('1 min shoaling index')
ax.set_title('1 min shoaling index averaged by episode')
ax.set_xticks(sorted(si_1min['minute_in_episode'].dropna().unique()))
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


In [ ]:
# Average the 1-minute attraction estimate within matched 10-minute experiment windows.
window_minutes = 10
si_windowed = si_1min.copy()

# Assign the full 5-minute episode to the 10-minute bin containing that episode's start.
si_windowed['episode_start_min'] = si_windowed['time_min'] - si_windowed['minute_in_episode']
si_windowed['experiment_window_start_min'] = (
    np.floor(si_windowed['episode_start_min'] / window_minutes) * window_minutes
).astype(int)
si_windowed['experiment_window_end_min'] = si_windowed['experiment_window_start_min'] + window_minutes

si_window_summary = summarize_plot_values(
    si_windowed,
    ['experiment_window_start_min', 'experiment_window_end_min', 'minute_in_episode', 'episode'],
    'si_1min',
).sort_values(['experiment_window_start_min', 'minute_in_episode'])

window_starts = sorted(si_window_summary['experiment_window_start_min'].dropna().unique())
n_cols = 4
n_rows = int(np.ceil(len(window_starts) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.4 * n_rows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

for ax, window_start in zip(axes, window_starts):
    window_data = si_window_summary.loc[si_window_summary['experiment_window_start_min'] == window_start]
    draw_lineplot_with_sem(
        ax,
        window_data,
        'minute_in_episode',
        'mean',
        'sem',
        split_by='episode',
        marker='o',
    )
    window_end = int(window_data['experiment_window_end_min'].iloc[0])
    ax.axhline(0, color='k', linestyle=':', linewidth=1)
    ax.set_title(f'{int(window_start)}-{window_end} min')
    ax.set_xticks(sorted(si_1min['minute_in_episode'].dropna().unique()))
    ax.set_xlabel('Minute within 5 min episode')
    ax.set_ylabel('1 min shoaling index')
    ax.legend(frameon=False, fontsize=8)

for ax in axes[len(window_starts):]:
    ax.set_visible(False)

fig.suptitle('1 min shoaling index averaged by episode and experiment window', y=1.02)
fig.tight_layout()
plt.show()


## Optional Carryover Check

In [ ]:
# Directly isolate the hypothesized carryover condition: continuous episodes after bout episodes.
continuous_after_bouts = asd_frame[
    (asd_frame['episode'] == '01k01f')
    & (asd_frame['prev_episode'] == '02k20f')
]
print(f'Continuous-after-bouts frame rows: {len(continuous_after_bouts):,}')

if not continuous_after_bouts.empty:
    fig, ax = plt.subplots(figsize=(6, 4))
    summary = summarize_plot_values(
        continuous_after_bouts,
        ['episode_frame', 'episode_time_min', GENOTYPE_COLUMN],
        'asd_mm',
    ).sort_values('episode_time_min')
    draw_lineplot_with_sem(ax, summary, 'episode_time_min', 'mean', 'sem', split_by=GENOTYPE_COLUMN)
    ax.set_title('01k01f')
    ax.set_xlabel('Episode time (min)')
    ax.set_ylabel('Animal-stimulus distance (mm)')
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()
